# mHC vs HC vs Baseline — Mini Colab T4 Training and Benchmark

Notebook này chạy **Mini controlled experiment** trên Google Colab T4 cho repo mHC: Manifold-Constrained Hyper-Connections.

Mục tiêu:
- Train from scratch ba biến thể: baseline Transformer residual, traditional HC, và mHC.
- Dùng cùng kiến trúc, cùng FineWeb10B shard budget, cùng optimizer, cùng effective batch, cùng số iteration.
- Báo cáo training loss, validation loss, validation perplexity, inference latency, throughput, và peak VRAM.
- Tạo checkpoint thật, validation metric thật, benchmark inference thật, bảng/biểu đồ sẵn đưa vào report.

**Honesty boundary:** Đây là controlled small-scale Colab T4 experiment, không phải full paper-scale reproduction. Không claim reproduce MMLU/BBH/GSM8K hoặc quality benchmark của paper.


## 0. Runtime requirement

Chọn runtime GPU trong Colab:

`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`

Nếu GPU không phải T4 vẫn chạy được, nhưng runtime/VRAM có thể khác report.


In [1]:
# Check GPU
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())


Wed May 20 13:05:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Mount Google Drive

Drive dùng để lưu checkpoints đã train, summaries, benchmark outputs, và figures. Training vẫn chạy local ở `/content/mhc` cho nhanh, sau đó notebook sync artifacts sang Drive.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Global configuration — Mini is report preset

`EXPERIMENT_PRESET = "mini"` là workflow chính để nộp/report.

Preset:
- `tiny`: smoke/debug only, không dùng làm kết quả chính.
- `mini`: main report experiment, default.
- `mini_plus`: stronger but slower, optional nếu còn thời gian/GPU.


In [3]:
from pathlib import Path
import os, json, shutil, subprocess, time, textwrap, glob, csv, math

# TODO: set your repo URL before cloning in Colab.
REPO_URL = "https://github.com/<your-username>/mHC-manifold-constrained-hyper-connections.git"
BRANCH = "main"  # or your working branch

DRIVE_ROOT = Path("/content/drive/MyDrive/mhc")
LOCAL_REPO = Path("/content/mhc")

EXPERIMENT_PRESET = "mini"  # "tiny", "mini", "mini_plus"

PRESETS = {
    "tiny": {
        "DATA_SHARDS": 1,
        "MAX_ITERS": 300,
        "EVAL_INTERVAL": 25,
        "EVAL_ITERS": 10,
        "BATCH_SIZE": 4,
        "GRAD_ACCUM": 4,
        "BLOCK_SIZE": 128,
        "N_LAYER": 2,
        "N_HEAD": 2,
        "N_EMBD": 128,
    },
    "mini": {
        "DATA_SHARDS": 1,
        "MAX_ITERS": 1500,
        "EVAL_INTERVAL": 75,
        "EVAL_ITERS": 20,
        "BATCH_SIZE": 2,
        "GRAD_ACCUM": 16,
        "BLOCK_SIZE": 256,
        "N_LAYER": 4,
        "N_HEAD": 4,
        "N_EMBD": 192,
    },
    "mini_plus": {
        "DATA_SHARDS": 3,
        "MAX_ITERS": 3000,
        "EVAL_INTERVAL": 100,
        "EVAL_ITERS": 30,
        "BATCH_SIZE": 2,
        "GRAD_ACCUM": 16,
        "BLOCK_SIZE": 256,
        "N_LAYER": 4,
        "N_HEAD": 4,
        "N_EMBD": 192,
    },
}

if EXPERIMENT_PRESET not in PRESETS:
    raise ValueError(f"unknown EXPERIMENT_PRESET={EXPERIMENT_PRESET!r}")

globals().update(PRESETS[EXPERIMENT_PRESET])

# Fixed runtime/training defaults.
DTYPE = "float16"
DEVICE = "cuda"
WANDB_LOG = "False"
COMPILE = "false"
COMPILE_MODEL = "False"
DATA_LOADER = "memmap"

# Inference benchmark defaults.
BENCH_BATCH_SIZE = 1
PROMPT_LEN = 128
GEN_LEN = 32
NUM_WARMUP = 5
NUM_ITERS = 20

RUN_NAME = f"t4-{EXPERIMENT_PRESET}-fineweb{DATA_SHARDS}shards-{MAX_ITERS}iters"
DRIVE_RUNS = DRIVE_ROOT / "runs" / RUN_NAME
DRIVE_REPORTS = DRIVE_ROOT / "reports" / RUN_NAME
LOG_DIR = DRIVE_REPORTS / "logs"

VARIANTS = {
    "baseline": {
        "config": "config/train_fineweb10B_mini_t4.py",
        "out_dir": "out-t4-mini-baseline",
        "expected": {"hc_disable": True, "mhc": False, "hc_num_streams": 1},
    },
    "hc": {
        "config": "config/train_fineweb10B_hc_mini_t4.py",
        "out_dir": "out-t4-mini-hc",
        "expected": {"hc_disable": False, "mhc": False, "hc_num_streams": 4},
    },
    "mhc": {
        "config": "config/train_fineweb10B_mhc_mini_t4.py",
        "out_dir": "out-t4-mini-mhc",
        "expected": {"hc_disable": False, "mhc": True, "hc_num_streams": 4},
    },
}

# Tiny preset uses tiny config files if selected, but mini remains report default.
if EXPERIMENT_PRESET == "tiny":
    VARIANTS["baseline"]["config"] = "config/train_fineweb10B_tiny_t4.py"
    VARIANTS["hc"]["config"] = "config/train_fineweb10B_hc_tiny_t4.py"
    VARIANTS["mhc"]["config"] = "config/train_fineweb10B_mhc_tiny_t4.py"
    VARIANTS["baseline"]["out_dir"] = "out-t4-tiny-baseline"
    VARIANTS["hc"]["out_dir"] = "out-t4-tiny-hc"
    VARIANTS["mhc"]["out_dir"] = "out-t4-tiny-mhc"
elif EXPERIMENT_PRESET == "mini_plus":
    VARIANTS["baseline"]["out_dir"] = "out-t4-mini-plus-baseline"
    VARIANTS["hc"]["out_dir"] = "out-t4-mini-plus-hc"
    VARIANTS["mhc"]["out_dir"] = "out-t4-mini-plus-mhc"

BASELINE_OUT = VARIANTS["baseline"]["out_dir"]
HC_OUT = VARIANTS["hc"]["out_dir"]
MHC_OUT = VARIANTS["mhc"]["out_dir"]
BASELINE_CONFIG = VARIANTS["baseline"]["config"]
HC_CONFIG = VARIANTS["hc"]["config"]
MHC_CONFIG = VARIANTS["mhc"]["config"]

print("selected preset:", EXPERIMENT_PRESET)
print("model shape:", f"L={N_LAYER}, H={N_HEAD}, D={N_EMBD}, block={BLOCK_SIZE}")
print("effective batch tokens/iter:", BATCH_SIZE * GRAD_ACCUM * BLOCK_SIZE)
print("dataset train shards:", DATA_SHARDS)
print("RUN_NAME:", RUN_NAME)
print("DRIVE_RUNS:", DRIVE_RUNS)
print("DRIVE_REPORTS:", DRIVE_REPORTS)
print("local out dirs:")
for name, spec in VARIANTS.items():
    print(f"  {name}: examples/nanogpt/{spec['out_dir']}")
print()
print("HONESTY: Mini is controlled small-scale Colab T4 experiment, not full paper reproduction.")
print("REPORT CLAIM: fair from-scratch baseline/HC/mHC comparison under same small training budget.")


selected preset: mini
model shape: L=4, H=4, D=192, block=256
effective batch tokens/iter: 8192
dataset train shards: 1
RUN_NAME: t4-mini-fineweb1shards-1500iters
DRIVE_RUNS: /content/drive/MyDrive/mhc/runs/t4-mini-fineweb1shards-1500iters
DRIVE_REPORTS: /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters
local out dirs:
  baseline: examples/nanogpt/out-t4-mini-baseline
  hc: examples/nanogpt/out-t4-mini-hc
  mhc: examples/nanogpt/out-t4-mini-mhc

HONESTY: Mini is controlled small-scale Colab T4 experiment, not full paper reproduction.
REPORT CLAIM: fair from-scratch baseline/HC/mHC comparison under same small training budget.


## 3. Clone repo and install dependencies

Nếu `REPO_URL` còn placeholder, sửa cell config trước. Nếu repo đã tồn tại trong `/content/mhc`, cell này pull branch hiện tại.


In [4]:
%cd /content

if str(REPO_URL).startswith("https://github.com/<"):
    raise ValueError("Set REPO_URL to your GitHub repo first.")

if LOCAL_REPO.exists():
    %cd /content/mhc
    !git fetch --all --prune
    !git checkout {BRANCH}
    !git pull --ff-only || true
else:
    !git clone --branch {BRANCH} {REPO_URL} /content/mhc
    %cd /content/mhc

!git status --short
!git rev-parse --short HEAD


/content


ValueError: Set REPO_URL to your GitHub repo first.

In [ ]:
%cd /content/mhc

# Install project + example dependencies. tqdm/matplotlib needed for progress/figures.
!python -m pip install -q -e ".[examples]"
!python -m pip install -q tqdm matplotlib pandas

# Fast sanity checks that do not train.
!python -m pytest -q tests/test_benchmark_summary.py tests/test_training_summary.py tests/test_t4_tiny_workflow.py


## 4. Download FineWeb10B GPT-2 shards

Default Mini uses `DATA_SHARDS = 1` train shard plus validation shard. This keeps Colab T4 disk/RAM pressure low.

Training uses `data_loader="memmap"`, so extra shards should not be concatenated into RAM. If you switch to eager loading, old extra shards can increase RAM usage.


In [ ]:
%cd /content/mhc

!python examples/nanogpt/data/fineweb10B/download.py {DATA_SHARDS}

!free -h
!du -sh examples/nanogpt/data/fineweb10B
!ls -lh examples/nanogpt/data/fineweb10B/*.bin | head


### Optional cleanup: reset FineWeb train shards

Run this only if you want to remove old train shards and keep exactly `DATA_SHARDS` train files. This can help if previous runs downloaded many shards.


In [ ]:
# Optional cleanup; disabled by default.
RESET_TRAIN_SHARDS = False

if RESET_TRAIN_SHARDS:
    data_dir = Path("/content/mhc/examples/nanogpt/data/fineweb10B")
    keep = {f"fineweb_train_{i:06d}.bin" for i in range(1, DATA_SHARDS + 1)}
    for path in data_dir.glob("fineweb_train_*.bin"):
        if path.name not in keep:
            print("removing", path)
            path.unlink()
    !ls -lh /content/mhc/examples/nanogpt/data/fineweb10B/*.bin | head
else:
    print("RESET_TRAIN_SHARDS=False; no files removed")


## 5. Smoke test — not for reporting

This cell only checks environment, data, model construction, memmap loader, checkpoint write, and summary write. Do **not** report these numbers.

Smoke values: `max_iters=20`, `eval_interval=5`, `eval_iters=2`, `batch_size=1`, `gradient_accumulation_steps=1`, `block_size=128`, `n_layer=2`, `n_head=2`, `n_embd=128`.


In [ ]:
%cd /content/mhc/examples/nanogpt

SMOKE_OUT = "out-smoke-tiny-baseline"

!python -u train.py config/train_fineweb10B_tiny_t4.py \
  "out_dir='{SMOKE_OUT}'" \
  max_iters=20 eval_interval=5 eval_iters=2 log_interval=1 \
  batch_size=1 gradient_accumulation_steps=1 \
  block_size=128 n_layer=2 n_head=2 n_embd=128 \
  "device='{DEVICE}'" "dtype='{DTYPE}'" \
  wandb_log=False compile_model=False data_loader='memmap' \
  use_tqdm=True heartbeat_interval_s=60

!ls -lh {SMOKE_OUT}
!cat {SMOKE_OUT}/summary.json


## 6. Helpers: train one variant, sync to Drive, verify outputs

Training logs stream live in Colab via `python -u train.py`. Each variant writes local logs and Drive logs. After each variant finishes, checkpoint/report artifacts sync to Drive and `verify_variant()` runs immediately.


In [ ]:
import subprocess, shutil, os, json, time
from pathlib import Path
from datetime import datetime

REPO = Path("/content/mhc")
NANOGPT = REPO / "examples" / "nanogpt"
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
DRIVE_REPORTS.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)


def run_cmd_live(cmd, *, cwd, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("command:", " ".join(cmd))
    print("log:", log_path)
    with log_path.open("a", encoding="utf-8") as log:
        log.write(f"\n===== start {datetime.now().isoformat(timespec='seconds')} =====\n")
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        code = proc.wait()
        log.write(f"===== end {datetime.now().isoformat(timespec='seconds')} code={code} =====\n")
    if code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {' '.join(cmd)}")


def sync_variant_to_drive(name):
    out_dir = NANOGPT / VARIANTS[name]["out_dir"]
    drive_dir = DRIVE_RUNS / VARIANTS[name]["out_dir"]
    if drive_dir.exists():
        shutil.rmtree(drive_dir)
    shutil.copytree(out_dir, drive_dir)
    print(f"synced {out_dir} -> {drive_dir}")


def restore_variant_from_drive(name):
    out_dir = NANOGPT / VARIANTS[name]["out_dir"]
    drive_dir = DRIVE_RUNS / VARIANTS[name]["out_dir"]
    if out_dir.exists() and (out_dir / "ckpt.pt").exists():
        return True
    if drive_dir.exists() and (drive_dir / "ckpt.pt").exists():
        if out_dir.exists():
            shutil.rmtree(out_dir)
        shutil.copytree(drive_dir, out_dir)
        print(f"restored {drive_dir} -> {out_dir}")
        return True
    return False


def verify_variant(name):
    spec = VARIANTS[name]
    out_dir = NANOGPT / spec["out_dir"]
    required = [
        "ckpt.pt",
        "summary.json",
        "config_effective.json",
        "dataset_manifest.json",
        "metrics.jsonl",
    ]
    missing = [item for item in required if not (out_dir / item).exists()]
    if missing:
        raise FileNotFoundError(f"{name} missing required artifacts: {missing}")
    summary = json.loads((out_dir / "summary.json").read_text())
    cfg = json.loads((out_dir / "config_effective.json").read_text())
    if summary.get("ok") is not True:
        raise RuntimeError(f"{name} summary ok is not True: {summary.get('error')}")
    expected_budget = {
        "max_iters": MAX_ITERS,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps_total": GRAD_ACCUM,
        "n_layer": N_LAYER,
        "n_head": N_HEAD,
        "n_embd": N_EMBD,
        "block_size": BLOCK_SIZE,
    }
    for key, expected in expected_budget.items():
        actual = cfg.get(key)
        if actual != expected:
            raise AssertionError(f"{name} config mismatch {key}: {actual} != {expected}")
    for key, expected in spec["expected"].items():
        actual = cfg.get(key)
        if actual != expected:
            raise AssertionError(f"{name} residual flag mismatch {key}: {actual} != {expected}")
    print(f"[OK] {name}: best_val_loss={summary.get('best_val_loss')} iter_num={summary.get('iter_num')}")
    return True


def train_variant(name, *, force=False):
    spec = VARIANTS[name]
    out_dir = NANOGPT / spec["out_dir"]
    log_path = LOG_DIR / f"{name}.log"

    if not force and restore_variant_from_drive(name):
        print(f"[SKIP] checkpoint exists for {name}: {out_dir / 'ckpt.pt'}")
        verify_variant(name)
        return

    if force and out_dir.exists():
        shutil.rmtree(out_dir)

    cmd = [
        "python", "-u", "train.py", spec["config"],
        f"out_dir='{spec['out_dir']}'",
        f"max_iters={MAX_ITERS}",
        f"eval_interval={EVAL_INTERVAL}",
        f"eval_iters={EVAL_ITERS}",
        "log_interval=5",
        f"batch_size={BATCH_SIZE}",
        f"gradient_accumulation_steps={GRAD_ACCUM}",
        f"block_size={BLOCK_SIZE}",
        f"n_layer={N_LAYER}",
        f"n_head={N_HEAD}",
        f"n_embd={N_EMBD}",
        f"device='{DEVICE}'",
        f"dtype='{DTYPE}'",
        f"wandb_log={WANDB_LOG}",
        "compile_model=False",
        f"data_loader='{DATA_LOADER}'",
        "use_tqdm=True",
        "heartbeat_interval_s=60",
    ]

    start = time.time()
    print(f"\n===== TRAIN {name} start {datetime.now().isoformat(timespec='seconds')} =====")
    run_cmd_live(cmd, cwd=NANOGPT, log_path=log_path)
    elapsed_min = (time.time() - start) / 60
    print(f"===== TRAIN {name} end {datetime.now().isoformat(timespec='seconds')} elapsed_min={elapsed_min:.1f} =====")
    verify_variant(name)
    sync_variant_to_drive(name)


## 7. Timeout / stop advice

If one Mini variant takes too long:

1. Stop runtime.
2. Set `MAX_ITERS = 500` or `1000` in preset/config cell.
3. Re-run all three variants with exactly same new budget.

Do **not** report mixed budgets like baseline=1500 and mHC=500. Fair comparison requires same architecture, same dataset shard count, same optimizer, same effective batch, same `max_iters`, same eval settings.


## 8. Main Mini Experiment: Train baseline, HC, and mHC

This is the main result to report.

All variants use same architecture/training budget. Only residual connection mechanism changes:
- baseline: standard Transformer residual
- HC: Hyper-Connections
- mHC: manifold-constrained Hyper-Connections


In [ ]:
%cd /content/mhc

for variant in ["baseline", "hc", "mhc"]:
    train_variant(variant, force=False)
    verify_variant(variant)
    

## 9. Summarize training runs

Creates:
- `training_summary.csv`
- `training_summary.md`
- `training_summary.json`

These include real validation loss, perplexity, checkpoint status, config, and run metadata.


In [ ]:
%cd /content/mhc

DRIVE_REPORTS.mkdir(parents=True, exist_ok=True)

!python examples/nanogpt/summarize_training_runs.py \
  --runs \
  "baseline=examples/nanogpt/{BASELINE_OUT}" \
  "hc=examples/nanogpt/{HC_OUT}" \
  "mhc=examples/nanogpt/{MHC_OUT}" \
  --output-dir "{DRIVE_REPORTS}"

!ls -lh "{DRIVE_REPORTS}"
!cat "{DRIVE_REPORTS}/training_summary.md"


## 10. Benchmark inference

Benchmark uses trained Mini checkpoints and synthetic token IDs. It measures runtime, not language quality:
- prefill latency
- full-context decode latency
- tokens/sec
- peak VRAM


In [ ]:
%cd /content/mhc

BENCH_DIR = DRIVE_REPORTS / "benchmarks"
BENCH_DIR.mkdir(parents=True, exist_ok=True)

BENCH_SPECS = {
    "baseline": (f"examples/nanogpt/{BASELINE_OUT}/ckpt.pt", f"examples/nanogpt/{BASELINE_CONFIG}"),
    "hc": (f"examples/nanogpt/{HC_OUT}/ckpt.pt", f"examples/nanogpt/{HC_CONFIG}"),
    "mhc": (f"examples/nanogpt/{MHC_OUT}/ckpt.pt", f"examples/nanogpt/{MHC_CONFIG}"),
}

for name, (ckpt, cfg) in BENCH_SPECS.items():
    if not Path(ckpt).exists():
        raise FileNotFoundError(f"missing checkpoint for benchmark: {ckpt}")
    run_cmd_live(
        [
            "python", "-u", "examples/nanogpt/benchmark_inference.py",
            "--ckpt", ckpt,
            "--config", cfg,
            "--device", DEVICE,
            "--dtype", DTYPE,
            "--batch-size", str(BENCH_BATCH_SIZE),
            "--prompt-len", str(PROMPT_LEN),
            "--gen-len", str(GEN_LEN),
            "--num-warmup", str(NUM_WARMUP),
            "--num-iters", str(NUM_ITERS),
            "--compile", COMPILE,
            "--output-json", str(BENCH_DIR / f"{name}.json"),
            "--output-csv", str(BENCH_DIR / f"{name}.csv"),
        ],
        cwd=REPO,
        log_path=LOG_DIR / f"benchmark-{name}.log",
    )

subprocess.run(["python", "examples/nanogpt/summarize_benchmarks.py", str(BENCH_DIR)], cwd=str(REPO), check=True)
print((BENCH_DIR / "summary.md").read_text())


## 11. Text generation demo

This is only a demo generation from the trained mHC Mini checkpoint. Model quality depends on small training budget, so do not overclaim text quality.


In [ ]:
%cd /content/mhc

!python examples/nanogpt/infer.py \
  --ckpt examples/nanogpt/{MHC_OUT}/ckpt.pt \
  --config examples/nanogpt/{MHC_CONFIG} \
  --device "{DEVICE}" \
  --dtype "{DTYPE}" \
  --prompt "The future of machine learning is" \
  --max-new-tokens 80 \
  --temperature 0.8 \
  --top-k 50


## 12. Visualization: report-ready charts

Charts are saved to `{DRIVE_REPORTS}/figures`:
- `train_loss_curve.png`
- `val_loss_curve.png`
- `best_val_loss_bar.png`
- `final_val_ppl_bar.png`
- `tokens_per_sec_curve.png`
- `peak_vram_curve.png`
- `inference_tokens_per_sec_bar.png`
- `decode_ms_per_token_bar.png`
- `benchmark_peak_vram_bar.png`


In [ ]:
%cd /content/mhc

FIG_DIR = DRIVE_REPORTS / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

!python examples/nanogpt/plot_training_comparison.py \
  --runs \
  "baseline=examples/nanogpt/{BASELINE_OUT}" \
  "hc=examples/nanogpt/{HC_OUT}" \
  "mhc=examples/nanogpt/{MHC_OUT}" \
  --benchmark-summary "{DRIVE_REPORTS}/benchmarks/summary.csv" \
  --output-dir "{FIG_DIR}"

!ls -lh "{FIG_DIR}"


## 13. How to report this experiment

Ready-to-say paragraph in Vietnamese:

> Do giới hạn tài nguyên Colab T4, em thiết kế một controlled small-scale experiment với cấu hình Mini. Ba biến thể baseline, HC và mHC được train from scratch với cùng kiến trúc, cùng dataset shard, cùng optimizer, cùng effective batch và cùng số iteration. Em so sánh bằng training loss, validation loss, validation perplexity, inference latency, throughput và peak VRAM. Kết quả này không nhằm reproduce toàn bộ benchmark của paper, mà nhằm kiểm chứng implementation và so sánh thực nghiệm trong điều kiện tài nguyên giới hạn.

Can claim:
- real training from scratch
- fair small-scale comparison
- validation loss / perplexity comparison
- inference runtime comparison

Must not claim:
- full paper reproduction
- MMLU/BBH/GSM8K results
- large-scale language model quality


## 14. Final artifact checklist

This verifies required files, summary status, same training budget, and expected residual flags.


In [ ]:
from pathlib import Path
import pandas as pd
import json

required_paths = [
    NANOGPT / BASELINE_OUT / "ckpt.pt",
    NANOGPT / HC_OUT / "ckpt.pt",
    NANOGPT / MHC_OUT / "ckpt.pt",
    DRIVE_REPORTS / "training_summary.csv",
    DRIVE_REPORTS / "training_summary.md",
    DRIVE_REPORTS / "training_summary.json",
    DRIVE_REPORTS / "benchmarks" / "summary.csv",
    DRIVE_REPORTS / "benchmarks" / "summary.md",
    DRIVE_REPORTS / "figures" / "train_loss_curve.png",
    DRIVE_REPORTS / "figures" / "val_loss_curve.png",
    DRIVE_REPORTS / "figures" / "best_val_loss_bar.png",
    DRIVE_REPORTS / "figures" / "final_val_ppl_bar.png",
    DRIVE_REPORTS / "figures" / "tokens_per_sec_curve.png",
    DRIVE_REPORTS / "figures" / "peak_vram_curve.png",
    DRIVE_REPORTS / "figures" / "inference_tokens_per_sec_bar.png",
    DRIVE_REPORTS / "figures" / "decode_ms_per_token_bar.png",
    DRIVE_REPORTS / "figures" / "benchmark_peak_vram_bar.png",
]

missing = []
for path in required_paths:
    ok = path.exists()
    print(("OK " if ok else "MISS"), path)
    if not ok:
        missing.append(str(path))
if missing:
    raise FileNotFoundError("missing required artifacts: " + ", ".join(missing))

summary = pd.read_csv(DRIVE_REPORTS / "training_summary.csv")
print(summary[["variant", "ok", "best_val_loss", "final_val_loss", "final_val_ppl"]])
if not summary["ok"].astype(str).eq("True").all():
    raise AssertionError("not all summary rows have ok=True")

same_keys = [
    "max_iters",
    "batch_size",
    "gradient_accumulation_steps",
    "n_layer",
    "n_head",
    "n_embd",
    "block_size",
]
for key in same_keys:
    values = set(summary[key].astype(str))
    print(key, values)
    if len(values) != 1:
        raise AssertionError(f"variants do not share same {key}: {values}")

expected_flags = {
    "baseline": {"hc_disable": "True", "mhc": "False", "hc_num_streams": "1"},
    "hc": {"hc_disable": "False", "mhc": "False", "hc_num_streams": "4"},
    "mhc": {"hc_disable": "False", "mhc": "True", "hc_num_streams": "4"},
}
for _, row in summary.iterrows():
    variant = row["variant"]
    for key, expected in expected_flags[variant].items():
        actual = str(row[key])
        print(variant, key, actual)
        if actual != expected:
            raise AssertionError(f"{variant} {key}: {actual} != {expected}")

print("FINAL CHECKLIST PASSED")
print("DRIVE_REPORTS:", DRIVE_REPORTS)
print("DRIVE_RUNS:", DRIVE_RUNS)


## 15. Optional Full T4 Experiment — not recommended for Colab Free

Old full-ish workflow used `block_size=1024`, `n_layer=6`, `n_head=6`, `n_embd=288`, `batch_size=8`, `gradient_accumulation_steps=8`, and thousands of iterations. On Colab T4 this can take many hours per variant.

This optional path is disabled by default. Use only if you have enough GPU time and still keep all variants on same budget.


In [ ]:
RUN_OPTIONAL_FULL = False

if RUN_OPTIONAL_FULL:
    run_cmd_live(
        [
            "bash", "examples/nanogpt/run_t4_full_compare.sh",
        ],
        cwd=REPO,
        log_path=LOG_DIR / "optional-full-t4.log",
    )
else:
    print("Optional full T4 experiment disabled. Mini is main report workflow.")
